# 01. text_embedders.py: текст, токены и conditioning

Пройдём от слов до cross-attention. Разберём CLIPTextEmbedder, Qwen2_5_VLTextEmbedder, Kandinsky5TextEmbedder и get_text_embedder. Для настоящих токенизаторов нужен интернет, но не 7B весов.


[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vladislav-vasilenko/generative-ai-research/blob/main/kandinsky-5/notebooks/01_text_embedders.ipynb)

### Подготовка Google Colab

Первая ячейка загружает код курса и устанавливает зависимости. Большие веса моделей она не скачивает. Для учебных опытов достаточно CPU; полная генерация в notebook 10 требует NVIDIA GPU. Если Colab попросит перезапуск сессии после установки, перезапустите и снова выполните ячейки сверху вниз.


In [ ]:
# Этот шаг нужен только в Google Colab; локально используется установленное окружение.
from pathlib import Path
import sys, subprocess
if 'google.colab' in sys.modules:
    checkout=Path('/content/generative-ai-research')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    course=checkout/'kandinsky-5'
    if not (course/'labkit').is_dir():
        # Пока PR не влит, берём его ветку. После merge используется main.
        subprocess.run(['git','-C',str(checkout),'fetch','--depth','1','origin','kandinsky-5-labs'],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach','FETCH_HEAD'],check=True)
    course=checkout/'kandinsky-5'
    expected={'torch':'2.8.0','transformers':'4.56.2','diffusers':'0.35.1'}
    loaded={name:str(getattr(sys.modules[name],'__version__','')) for name in expected if name in sys.modules}
    subprocess.run([sys.executable,'-m','pip','install','-r',str(course/'requirements-labs.txt')],check=True)
    if any(version.split('+')[0]!=expected[name] for name,version in loaded.items()):
        raise RuntimeError('Библиотека уже была импортирована до установки. Перезапустите сессию Colab и выполните notebook с первой ячейки.')
    if str(course) not in sys.path: sys.path.insert(0,str(course))
    print('Course:',course)
    print('Git revision:',subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],text=True).strip())
else:
    print('Локальный запуск: зависимости устанавливаются по README.')


### Перед началом

Запускайте ячейки сверху вниз. Для каждой операции сначала прочитайте пояснение, затем выполните код и сравните вывод с ожидаемым. Установка описана в README; сетевых загрузок в учебном режиме нет.


In [ ]:
from pathlib import Path
import sys, os
# В Colab загрузите архив KandinskyLab-colab.zip через панель Files и распакуйте
# ячейкой из README. Здесь ищем корень курса, не меняя рабочую директорию.
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/KandinskyLab'), Path('/content/generative-ai-research/kandinsky-5')]
ROOT=next((p for p in candidates if (p/'labkit').is_dir()), None)
if ROOT is None: raise RuntimeError('Сначала распакуйте архив курса; инструкции в README.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
os.environ.setdefault('MPLCONFIGDIR',str(ROOT/'artifacts/matplotlib'))
import torch, numpy as np, matplotlib.pyplot as plt
from labkit.common import SOURCE, REVISION, device_info, source_excerpt, release, shifted_schedule
# Маленькие эксперименты по умолчанию CPU: быстро и одинаково в Mac/Colab.
# Для отдельной проверки Metal смените LAB_DEVICE на mps.
LAB_DEVICE='cpu'
torch.manual_seed(42)
torch.set_num_threads(2)
print(device_info())


### 1. Токен — индекс, embedding — вектор

В учебной таблице одно слово равно одному токену. Настоящие Qwen/CLIP используют подслова: русское слово может занимать несколько индексов. Таблица Embedding делает lookup, а Transformer затем добавляет контекст. Следующий пример демонстрирует только lookup.


In [ ]:
vocab={'<pad>':0,'кот':1,'красная':2,'шапка':3}
ids=torch.tensor([[1,2,3,0]])
lookup=torch.nn.Embedding(4,6)
print(ids.shape, '→',lookup(ids).shape)
print('padding mask:',ids!=0)


### 2. Две ветви conditioning

Qwen даёт последовательность [B,L,3584] для деталей описания. CLIP даёт pooled [B,768] для общего контекста и смешивания с time embedding. Это `pooler_output`, а не обязательно нормализованное `get_text_features()`. У них разные vocabularies и способы pooling.


In [ ]:
source_excerpt('kandinsky/models/text_embedders.py','class ClipTextEmbedder',29)


### 3. Prompt template и crop

Upstream добавляет системный текст перед пользовательским запросом. Для video отрезаются первые 129 hidden-state позиций, для image — 41. Это фиксированные константы конкретной реализации. Нельзя незаметно сменить tokenizer или шаблон: тогда crop будет означать другой текст.


In [ ]:
from labkit.components import prompt_template
template,crop=prompt_template('video')
print('crop:',crop); print(template.format('Кот медленно поворачивает голову.'))
source_excerpt('kandinsky/models/text_embedders.py','max_length = self.max_length + crop_start',24)


### 4. Почему mask и padding важны

Padding выравнивает длины. В bool-маске SDPA True означает разрешённый ключ. Ниже усреднение с mask исключает фиктивные позиции. Это иллюстрация, а не реальный pooling CLIP. Upstream передаёт mask в DiT; внутри Qwen T2V-вызов использует только input_ids — воспроизводим именно его.


In [ ]:
values=torch.tensor([[[1.,2.],[3.,4.],[100.,100.]]])
mask=torch.tensor([[True,True,False]])
pooled=(values*mask[...,None]).sum(1)/mask.sum(1,keepdim=True)
print('без mask:',values.mean(1),'с mask:',pooled)


### 5. Настоящие токенизаторы без больших весов

Включите RUN_TOKENIZERS, чтобы скачать только tokenizer/processor файлы. Сравним русский и английский запросы, количество токенов и truncation. Сначала читаем длины без обрезки; иначе потерю хвоста трудно заметить.


In [ ]:
RUN_TOKENIZERS=False
if RUN_TOKENIZERS:
    from huggingface_hub import snapshot_download
    from transformers import AutoProcessor, CLIPTokenizer
    from labkit.components import REVISIONS
    qrepo='Qwen/Qwen2.5-VL-7B-Instruct'; crepo='openai/clip-vit-large-patch14'
    qp=snapshot_download(qrepo,revision=REVISIONS[qrepo],allow_patterns=['*.json','*.txt','*.jinja','*.model'])
    cp=snapshot_download(crepo,revision=REVISIONS[crepo],allow_patterns=['*.json','*.txt'])
    q=AutoProcessor.from_pretrained(qp,use_fast=True).tokenizer
    c=CLIPTokenizer.from_pretrained(cp)
    for text in ['Кот в красной шапке','A cat in a red hat','Кот '*400]:
        for name,tok in [('Qwen',q),('CLIP',c)]:
            token_ids=tok(text,truncation=False)['input_ids']
            print(name,len(token_ids),tok.convert_ids_to_tokens(token_ids[:12]))
else: print('Настоящие токенизаторы пропущены; веса не скачивались.')


### 6. Настоящие embeddings отдельно от DiT

RUN_ENCODERS загружает Qwen, выполняет один forward, удаляет его, затем загружает CLIP. Большие модели не живут одновременно. Сохранённый conditioning можно повторно использовать для разных seed и sampler, пока текст, шаблон, crop и версии энкодеров совпадают.


In [ ]:
RUN_ENCODERS=False
if RUN_ENCODERS:
    from labkit.components import download_component,encode_conditioning
    for name in ['qwen','clip']: download_component(name)
    device=device_info()['device']
    conditioning=encode_conditioning('A cat slowly turns its head.',device=device)
    for key,value in conditioning.items(): print(key,value.shape,value.dtype)
    torch.save(conditioning,ROOT/'artifacts/conditioning.pt')
else: print('7B энкодер не запускался.')


### 7. Prompt expansion — отдельное действие

Метод expand_text_prompt использует `.generate()` для переписывания запроса. Основной encode использует hidden states одного forward. Это разные задачи и разная latency. Для контролируемого эксперимента отключаем expansion и сохраняем точный запрос.


In [ ]:
source_excerpt('kandinsky/models/text_embedders.py','class Kandinsky5TextEmbedder',30)


#


In [ ]:
#


### Проверьте понимание

Не меняйте сразу несколько параметров. Сначала сформулируйте гипотезу, затем измените один параметр и сравните результат с исходным. Запишите: что было входом, что стало выходом, какие размеры изменились и почему.

Учебные модели со случайными весами показывают **механизм**, но не качество Kandinsky. Тяжёлые эксперименты запускаются только при включении соответствующего флага; пропуск этих ячеек не означает успешный запуск настоящих весов.
